In [1]:
#1: Install Required Libraries
!pip install scikit-learn pandas numpy matplotlib seaborn plotly
!pip install nltk textblob wordcloud
!pip install -q ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.6/27.6 MB 4.8 MB/s eta 0:00:00m eta 0:00:010:00:01
  Installing build dependenciedone
  Getting requirements to build wheel ... one
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.1 MB/s eta 0:00:005.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.3/624.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.3/797.3 kB 5.3 MB/s eta 0:00:00
  Created wheel for wordcloud: filename=wordcloud-1.9.4-cp312-cp312-linux_aarch64.whl size=182154 sha256=2dc5a77c62f9109ab7328af1488e5b60eba35e5548da51bdee0c9fd746da4429
  Stored in directory: /home/cis/.cache/pip/wheels/93/2a/62/2b71344178dc0164dcf96a0449d00886229f8e050ffa76a010
Successfully built wordcloud
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [textblob]━━━━━━━━━ 4/6 [wordcloud]


In [2]:
#2: Import Libraries and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
import re
from textblob import TextBlob
from wordcloud import WordCloud
from collections import Counter

# Download NLTK data
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/vader_lexicon')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('vader_lexicon')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

# Machine Learning Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("📚 Libraries imported successfully!")
print("🔍 NLTK components downloaded and ready!")

[nltk_data] Downloading package punkt to /home/cis/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /home/cis/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/cis/nltk_data...


📚 Libraries imported successfully!
🔍 NLTK components downloaded and ready!


In [3]:
#3: Generate Synthetic Social Media Dataset
# แก้ไขปัญหา ValueError: probabilities do not sum to 1
# ใช้โค้ดนี้แทนที่ใน Cell 3 ของ notebook

import numpy as np
import pandas as pd
import re

#3: Generate Synthetic Social Media Dataset
# Generate synthetic social media emotion dataset
def generate_emotion_dataset(n_samples=5000):
    """
    Generate synthetic social media posts with emotions
    Emotions: Happy, Sad, Angry, Fear, Surprise, Neutral
    """
    np.random.seed(42)
    
    # Define emotion templates and patterns
    emotion_templates = {
        'Happy': [
            "I'm so excited about {activity}! This is amazing! 😊 #{hashtag}",
            "Just had the best {time_period}! Feeling grateful and blessed! ✨",
            "Love spending time with {people}! Life is beautiful! 💕",
            "Finally achieved my goal! So proud of myself! 🎉 #{hashtag}",
            "What a wonderful {event}! Can't stop smiling! 😄",
            "Great news everyone! Just got {achievement}! Celebrating! 🎊",
            "Perfect weather for {activity}! Feeling fantastic! ☀️",
            "Thank you all for the birthday wishes! Best day ever! 🎂"
        ],
        'Sad': [
            "Feeling really down today... miss {person} so much 😢",
            "Why does everything have to be so difficult? Having a tough {time_period}",
            "Just heard about {bad_news}... my heart is broken 💔",
            "Can't believe {person} is no longer with us... RIP 😭",
            "Feeling lonely and lost... need some comfort right now",
            "Failed my {exam/interview}... feeling so disappointed in myself",
            "Rainy days always make me feel melancholic... #{hashtag}",
            "Missing the good old {time_period}... things were simpler then"
        ],
        'Angry': [
            "I can't believe how {adjective} this {situation} is! So frustrated! 😡",
            "Tired of people being {negative_trait}! Had enough! #{hashtag}",
            "This {service} is absolutely terrible! Worst experience ever! 🤬",
            "Why do some people have to be so {negative_trait}?! Makes me furious!",
            "Stuck in traffic for {time}! This is ridiculous! 😤",
            "Customer service was horrible today! Never going back! #angry",
            "Politicians need to stop making {type} promises! Fed up!",
            "My {device} broke again! Such poor quality! Waste of money!"
        ],
        'Fear': [
            "Really nervous about {future_event}... what if things go wrong? 😰",
            "Can't sleep... too worried about {concern} happening",
            "Scared about the {situation}... hope everything will be okay 😨",
            "Anxious about {upcoming_event}... need to calm down somehow",
            "Hearing about {scary_news} makes me so worried about the future",
            "What if I fail {important_task}? So much pressure... 😟",
            "Feeling overwhelmed by all the {problems} lately... need help",
            "Can't shake this feeling that something bad might happen..."
        ],
        'Surprise': [
            "OMG! I can't believe {unexpected_event} just happened! 😲",
            "Wait, what?! Did {person} really just {action}?! Mind blown! 🤯",
            "Plot twist! Turns out {revelation}! Never saw that coming!",
            "Holy cow! Just found out {news}! This is incredible! 😱",
            "No way! {unexpected_outcome}! Still processing this... #{hashtag}",
            "Didn't expect {event} to happen today! Life is full of surprises!",
            "Whoa! {discovery} completely changed my perspective! Amazing!",
            "Shocked to learn that {fact}! How did I not know this before?!"
        ],
        'Neutral': [
            "Just finished {routine_activity}. Now heading to {place}.",
            "Weather is {weather_condition} today. Perfect for {activity}.",
            "Reading about {topic}. Quite interesting information here.",
            "Meeting with {people} at {time}. Should be productive.",
            "Trying out this new {product/service}. So far so good.",
            "Working on {project} today. Making steady progress.",
            "Had {meal} for {meal_time}. Standard {day} routine.",
            "Checking out {place} this {time_period}. Heard good things about it."
        ]
    }
    
    # Replacement words for templates
    replacements = {
        'activity': ['hiking', 'cooking', 'reading', 'shopping', 'exercising', 'traveling', 'gaming', 'studying'],
        'hashtag': ['blessed', 'grateful', 'excited', 'happy', 'goodvibes', 'positive', 'amazing', 'wonderful'],
        'time_period': ['day', 'week', 'weekend', 'vacation', 'month', 'morning', 'evening', 'afternoon'],
        'people': ['family', 'friends', 'colleagues', 'teammates', 'neighbors', 'classmates', 'siblings', 'cousins'],
        'event': ['party', 'concert', 'wedding', 'graduation', 'conference', 'festival', 'celebration', 'reunion'],
        'achievement': ['promoted', 'accepted', 'scholarship', 'new job', 'award', 'recognition', 'diploma', 'certificate'],
        'person': ['grandma', 'grandpa', 'friend', 'pet', 'mentor', 'colleague', 'neighbor', 'teacher'],
        'bad_news': ['accident', 'illness', 'loss', 'disaster', 'crisis', 'emergency', 'tragedy', 'setback'],
        'exam/interview': ['exam', 'interview', 'presentation', 'audition', 'test', 'evaluation', 'assessment', 'review'],
        'adjective': ['unfair', 'ridiculous', 'absurd', 'unreasonable', 'outrageous', 'unacceptable', 'terrible', 'awful'],
        'situation': ['policy', 'decision', 'rule', 'system', 'process', 'procedure', 'requirement', 'regulation'],
        'negative_trait': ['rude', 'selfish', 'dishonest', 'inconsiderate', 'arrogant', 'ignorant', 'lazy', 'mean'],
        'service': ['restaurant', 'hotel', 'airline', 'store', 'company', 'website', 'app', 'delivery'],
        'time': ['2 hours', '3 hours', '45 minutes', '1 hour', '90 minutes', 'forever', 'way too long', 'ages'],
        'device': ['phone', 'laptop', 'car', 'tablet', 'headphones', 'TV', 'computer', 'camera'],
        'future_event': ['job interview', 'exam', 'presentation', 'surgery', 'meeting', 'date', 'trip', 'move'],
        'concern': ['worst case scenario', 'failure', 'rejection', 'embarrassment', 'mistake', 'accident', 'loss', 'conflict'],
        'scary_news': ['pandemic', 'economic crisis', 'natural disaster', 'violence', 'war', 'climate change', 'recession', 'outbreak'],
        'important_task': ['this project', 'the interview', 'my exam', 'the presentation', 'this opportunity', 'my goal', 'this challenge', 'the test'],
        'problems': ['work stress', 'family issues', 'health concerns', 'financial pressure', 'relationship problems', 'career decisions', 'life changes', 'responsibilities'],
        'unexpected_event': ['surprise party', 'job offer', 'lottery win', 'celebrity sighting', 'old friend visit', 'proposal', 'pregnancy news', 'inheritance'],
        'action': ['announce retirement', 'get married', 'move abroad', 'change career', 'start business', 'adopt pet', 'buy house', 'have baby'],
        'revelation': ['they were twins', 'it was fake', 'they knew all along', 'it was planned', 'they were related', 'it was true', 'they lied', 'it worked'],
        'news': ['celebrity engagement', 'company merger', 'scientific discovery', 'sports upset', 'political announcement', 'technology breakthrough', 'space mission', 'medical breakthrough'],
        'unexpected_outcome': ['team won championship', 'stock price doubled', 'weather cleared up', 'plan worked perfectly', 'problem solved itself', 'help arrived', 'opportunity appeared', 'miracle happened'],
        'discovery': ['family secret', 'hidden talent', 'old photos', 'lost item', 'new feature', 'shortcut route', 'perfect solution', 'amazing fact'],
        'fact': ['earth history', 'animal behavior', 'space phenomena', 'human psychology', 'cultural tradition', 'scientific principle', 'historical event', 'technology evolution'],
        'routine_activity': ['morning jog', 'grocery shopping', 'office work', 'house cleaning', 'meal prep', 'email checking', 'commuting', 'studying'],
        'place': ['office', 'gym', 'store', 'park', 'library', 'cafe', 'home', 'school'],
        'weather_condition': ['sunny', 'cloudy', 'rainy', 'windy', 'foggy', 'clear', 'humid', 'cool'],
        'topic': ['technology', 'history', 'science', 'politics', 'health', 'environment', 'economics', 'culture'],
        'product/service': ['restaurant', 'app', 'book', 'movie', 'course', 'workout', 'recipe', 'method'],
        'project': ['work assignment', 'home renovation', 'garden project', 'research paper', 'business plan', 'creative project', 'learning goal', 'fitness plan'],
        'meal': ['breakfast', 'lunch', 'dinner', 'snack', 'coffee', 'smoothie', 'salad', 'soup'],
        'meal_time': ['breakfast', 'lunch', 'dinner', 'brunch', 'supper']
    }
    
    # Generate posts
    posts = []
    emotions = []
    additional_features = []
    
    emotion_distribution = [0.25, 0.15, 0.15, 0.10, 0.10, 0.25]  # Happy, Sad, Angry, Fear, Surprise, Neutral
    emotion_list = list(emotion_templates.keys())
    
    for i in range(n_samples):
        # Select emotion based on distribution
        emotion = np.random.choice(emotion_list, p=emotion_distribution)
        
        # Select template
        template = np.random.choice(emotion_templates[emotion])
        
        # Replace placeholders
        post = template
        for placeholder, options in replacements.items():
            if '{' + placeholder + '}' in post:
                replacement = np.random.choice(options)
                post = post.replace('{' + placeholder + '}', replacement)
        
        # Add some natural variations
        if np.random.random() < 0.3:  # 30% chance to add extra emoji
            emoji_map = {
                'Happy': ['😊', '😄', '🎉', '❤️', '✨', '🌟', '💕', '🥰'],
                'Sad': ['😢', '😭', '💔', '😞', '😔', '😰', '😟', '🥺'],
                'Angry': ['😡', '🤬', '😤', '👿', '💢', '😠', '🔥', '⚡'],
                'Fear': ['😰', '😨', '😱', '😟', '🙁', '😳', '😬', '😓'],
                'Surprise': ['😲', '😱', '🤯', '😮', '😯', '👀', '‼️', '❗'],
                'Neutral': ['🤔', '💭', '📝', '📊', '🔍', '💡', '📚', '⭐']
            }
            extra_emoji = np.random.choice(emoji_map[emotion])
            post += f" {extra_emoji}"
        
        if np.random.random() < 0.2:  # 20% chance to add "lol", "haha", etc.
            filler_words = {
                'Happy': ['haha', 'lol', 'yay', 'woohoo', 'awesome'],
                'Sad': ['sigh', 'ugh', 'meh', 'unfortunately', 'sadly'],
                'Angry': ['seriously', 'ugh', 'omg', 'unbelievable', 'ridiculous'],
                'Fear': ['hopefully', 'fingers crossed', 'pray', 'wish', 'please'],
                'Surprise': ['wow', 'omg', 'whoa', 'incredible', 'unbelievable'],
                'Neutral': ['btw', 'anyway', 'so', 'well', 'apparently']
            }
            filler = np.random.choice(filler_words[emotion])
            post = filler.capitalize() + ", " + post.lower()
        
        posts.append(post)
        emotions.append(emotion)
        
        # Generate additional features
        text_length = len(post)
        word_count = len(post.split())
        emoji_count = len(re.findall(r'[😀-🿿]', post))
        hashtag_count = post.count('#')
        mention_count = post.count('@')
        exclamation_count = post.count('!')
        question_count = post.count('?')
        caps_ratio = sum(1 for c in post if c.isupper()) / len(post) if len(post) > 0 else 0
        
        # Time features (simulate posting patterns) - FIXED VERSION
        hour_probabilities = [0.02, 0.01, 0.01, 0.01, 0.01, 0.02, 0.03, 0.05, 
                             0.06, 0.07, 0.08, 0.09, 0.09, 0.08, 0.07, 0.06, 
                             0.05, 0.04, 0.05, 0.06, 0.07, 0.06, 0.04, 0.03]
        # Normalize probabilities to sum to 1
        hour_probabilities = np.array(hour_probabilities)
        hour_probabilities = hour_probabilities / hour_probabilities.sum()
        hour = np.random.choice(range(24), p=hour_probabilities)
        day_of_week = np.random.randint(0, 7)
        
        # Platform simulation
        platform = np.random.choice(['Twitter', 'Instagram', 'Facebook'], p=[0.4, 0.35, 0.25])
        
        additional_features.append({
            'text_length': text_length,
            'word_count': word_count,
            'emoji_count': emoji_count,
            'hashtag_count': hashtag_count,
            'mention_count': mention_count,
            'exclamation_count': exclamation_count,
            'question_count': question_count,
            'caps_ratio': caps_ratio,
            'hour': hour,
            'day_of_week': day_of_week,
            'platform': platform
        })
    
    # Create DataFrame
    df = pd.DataFrame({
        'post_text': posts,
        'emotion': emotions
    })
    
    # Add additional features
    feature_df = pd.DataFrame(additional_features)
    df = pd.concat([df, feature_df], axis=1)
    
    return df

# Generate dataset
print("🎭 Generating Social Media Emotion Dataset...")
social_df = generate_emotion_dataset(5000)

print("✅ Social Media Dataset Generated Successfully!")
print(f"Dataset Shape: {social_df.shape}")
print(f"Emotions: {social_df['emotion'].unique()}")
print(f"Total Posts: {len(social_df)}")

# Display sample posts
print("\n📱 SAMPLE POSTS BY EMOTION:")
print("=" * 50)
for emotion in social_df['emotion'].unique():
    sample_post = social_df[social_df['emotion'] == emotion]['post_text'].iloc[0]
    print(f"\n{emotion.upper()}:")
    print(f"  \"{sample_post}\"") 

🎭 Generating Social Media Emotion Dataset...
✅ Social Media Dataset Generated Successfully!
Dataset Shape: (5000, 13)
Emotions: [np.str_('Sad') np.str_('Angry') np.str_('Happy') np.str_('Neutral')
 np.str_('Fear') np.str_('Surprise')]
Total Posts: 5000

📱 SAMPLE POSTS BY EMOTION:

SAD:
  "Feeling lonely and lost... need some comfort right now 🥺"

ANGRY:
  "Ugh, stuck in traffic for 1 hour! this is ridiculous! 😤 😤"

HAPPY:
  "What a wonderful party! Can't stop smiling! 😄"

NEUTRAL:
  "Weather is sunny today. Perfect for cooking. 💭"

FEAR:
  "Hearing about recession makes me so worried about the future"

SURPRISE:
  "Incredible, shocked to learn that historical event! how did i not know this before?!"


In [ ]:
#4: Data Exploration - Basic Information
print("🔍 SOCIAL MEDIA DATA EXPLORATION")
print("=" * 50)

# Basic dataset information
print("📊 DATASET OVERVIEW:")
print(f"Total Posts: {len(social_df):,}")
print(f"Features: {social_df.shape[1]}")
print(f"Emotion Classes: {social_df['emotion'].nunique()}")
print(f"Average Post Length: {social_df['text_length'].mean():.1f} characters")
print(f"Average Word Count: {social_df['word_count'].mean():.1f} words")

# Display first few rows
print("\n📱 FIRST 5 POSTS:")
print("=" * 60)
for i in range(5):
    row = social_df.iloc[i]
    print(f"\nPost {i+1} [{row['emotion'].upper()}]:")
    print(f"Text: \"{row['post_text']}\"")
    print(f"Length: {row['text_length']} chars, Words: {row['word_count']}, Emojis: {row['emoji_count']}")

# Dataset info
print("\n📋 DATASET INFO:")
print("=" * 30)
print(social_df.info())

# Emotion distribution
print("\n🎭 EMOTION DISTRIBUTION:")
print("=" * 35)
emotion_counts = social_df['emotion'].value_counts()
print(emotion_counts)

print(f"\nPercentage Distribution:")
for emotion, count in emotion_counts.items():
    percentage = (count / len(social_df)) * 100
    print(f"{emotion}: {percentage:.1f}%")

# Statistical summary of numerical features
print("\n📊 NUMERICAL FEATURES SUMMARY:")
print("=" * 40)
numerical_cols = ['text_length', 'word_count', 'emoji_count', 'hashtag_count', 
                 'mention_count', 'exclamation_count', 'question_count', 'caps_ratio']
print(social_df[numerical_cols].describe())

In [ ]:
#5: Emotion Distribution Visualization
# Emotion distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Emotion count bar chart
emotion_counts = social_df['emotion'].value_counts()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#D63031']
bars = axes[0,0].bar(emotion_counts.index, emotion_counts.values, color=colors)
axes[0,0].set_title('🎭 Emotion Distribution (Count)', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Emotion')
axes[0,0].set_ylabel('Number of Posts')
axes[0,0].tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[0,0].text(bar.get_x() + bar.get_width()/2., height + 10,
                  f'{int(height)}',
                  ha='center', va='bottom', fontweight='bold')

# 2. Emotion pie chart
wedges, texts, autotexts = axes[0,1].pie(emotion_counts.values, 
                                        labels=emotion_counts.index,
                                        colors=colors,
                                        autopct='%1.1f%%',
                                        startangle=90)
axes[0,1].set_title('🥧 Emotion Distribution (Percentage)', fontsize=14, fontweight='bold')

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# 3. Platform distribution by emotion
platform_emotion = pd.crosstab(social_df['platform'], social_df['emotion'])
platform_emotion.plot(kind='bar', ax=axes[1,0], color=colors, stacked=True)
axes[1,0].set_title('📱 Platform Distribution by Emotion', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('Platform')
axes[1,0].set_ylabel('Number of Posts')
axes[1,0].legend(title='Emotion', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Hourly posting pattern by emotion
hour_emotion = social_df.groupby(['hour', 'emotion']).size().unstack(fill_value=0)
hour_emotion.plot(kind='line', ax=axes[1,1], color=colors, marker='o', alpha=0.8)
axes[1,1].set_title('⏰ Hourly Posting Patterns by Emotion', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Hour of Day')
axes[1,1].set_ylabel('Number of Posts')
axes[1,1].legend(title='Emotion', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].set_xticks(range(0, 24, 4))

plt.tight_layout()
plt.show()

# Summary insights
print("💡 KEY INSIGHTS FROM EMOTION DISTRIBUTION:")
print("=" * 50)
print(f"📊 Most Common Emotion: {emotion_counts.index[0]} ({emotion_counts.values[0]} posts)")
print(f"📊 Least Common Emotion: {emotion_counts.index[-1]} ({emotion_counts.values[-1]} posts)")

# Platform insights
platform_stats = social_df.groupby('platform')['emotion'].value_counts()
print(f"\n📱 Platform Usage:")
for platform in social_df['platform'].unique():
    platform_data = social_df[social_df['platform'] == platform]
    dominant_emotion = platform_data['emotion'].value_counts().index[0]
    print(f"  {platform}: {len(platform_data)} posts, dominant emotion: {dominant_emotion}")

# Time insights
peak_hour = social_df['hour'].value_counts().index[0]
print(f"\n⏰ Peak Posting Hour: {peak_hour}:00")

In [ ]:
#6: Text Analysis by Emotion
# Text analysis by emotion
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

emotions = social_df['emotion'].unique()
axes = axes.flatten()

# Text length distribution by emotion
for i, emotion in enumerate(emotions):
    emotion_data = social_df[social_df['emotion'] == emotion]
    axes[i].hist(emotion_data['text_length'], bins=20, alpha=0.7, 
                color=colors[i], edgecolor='black')
    axes[i].set_title(f'📝 {emotion} - Text Length Distribution', fontweight='bold')
    axes[i].set_xlabel('Text Length (characters)')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(emotion_data['text_length'].mean(), color='red', 
                   linestyle='--', linewidth=2, label=f'Mean: {emotion_data["text_length"].mean():.0f}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Feature comparison across emotions
print("📊 TEXT FEATURES BY EMOTION:")
print("=" * 50)

feature_summary = social_df.groupby('emotion')[numerical_cols].mean()
print(feature_summary.round(2))

# Box plots for key features
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Text length
sns.boxplot(data=social_df, x='emotion', y='text_length', palette=colors, ax=axes[0,0])
axes[0,0].set_title('📏 Text Length by Emotion', fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=45)

# Word count
sns.boxplot(data=social_df, x='emotion', y='word_count', palette=colors, ax=axes[0,1])
axes[0,1].set_title('📝 Word Count by Emotion', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=45)

# Emoji count
sns.boxplot(data=social_df, x='emotion', y='emoji_count', palette=colors, ax=axes[1,0])
axes[1,0].set_title('😊 Emoji Count by Emotion', fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=45)

# Caps ratio
sns.boxplot(data=social_df, x='emotion', y='caps_ratio', palette=colors, ax=axes[1,1])
axes[1,1].set_title('🔠 Capital Letters Ratio by Emotion', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Statistical insights
print("\n💡 TEXT ANALYSIS INSIGHTS:")
print("=" * 40)

for emotion in emotions:
    emotion_data = social_df[social_df['emotion'] == emotion]
    print(f"\n{emotion.upper()}:")
    print(f"  Average Length: {emotion_data['text_length'].mean():.1f} characters")
    print(f"  Average Words: {emotion_data['word_count'].mean():.1f}")
    print(f"  Average Emojis: {emotion_data['emoji_count'].mean():.1f}")
    print(f"  Caps Ratio: {emotion_data['caps_ratio'].mean():.3f}")
    print(f"  Exclamations: {emotion_data['exclamation_count'].mean():.1f}")

# Find most distinctive features for each emotion
print(f"\n🔍 MOST DISTINCTIVE TEXT FEATURES:")
print("=" * 40)

overall_means = social_df[numerical_cols].mean()
for emotion in emotions:
    emotion_data = social_df[social_df['emotion'] == emotion]
    emotion_means = emotion_data[numerical_cols].mean()
    
    # Calculate relative differences
    differences = (emotion_means - overall_means) / overall_means
    
    # Find most distinctive (highest positive difference)
    most_distinctive = differences.abs().idxmax()
    difference_value = differences[most_distinctive]
    
    print(f"{emotion}: Most distinctive feature is '{most_distinctive}' "
          f"({difference_value:+.1%} vs average)")

In [ ]:
#7: Text Preprocessing and Feature Extraction
import string
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

print("🔤 TEXT PREPROCESSING AND FEATURE EXTRACTION")
print("=" * 60)

# Text preprocessing function
def preprocess_text(text):
    """
    Comprehensive text preprocessing for social media posts
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove mentions and hashtags (keep the content)
    text = re.sub(r'[@#]\w+', '', text)
    
    # Keep emojis but add spaces around them for tokenization
    text = re.sub(r'([😀-🿿])', r' \1 ', text)
    
    # Remove extra punctuation (but keep some for sentiment)
    text = re.sub(r'[^\w\s😀-🿿!?.,]', '', text)
    
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Strip whitespace
    text = text.strip()
    
    return text

# Apply preprocessing
print("🧹 Preprocessing text data...")
social_df['processed_text'] = social_df['post_text'].apply(preprocess_text)

print("✅ Text preprocessing completed!")

# Show examples of preprocessing
print("\n📝 PREPROCESSING EXAMPLES:")
print("=" * 50)
for i in range(3):
   print(f"\nExample {i+1}:")
   print(f"Original: \"{social_df['post_text'].iloc[i]}\"")
   print(f"Processed: \"{social_df['processed_text'].iloc[i]}\"")

# Text feature extraction
print("\n🔍 EXTRACTING TEXT FEATURES...")
print("=" * 40)

# 1. TF-IDF Features
print("📊 Creating TF-IDF features...")
tfidf_vectorizer = TfidfVectorizer(
   max_features=1000,  # Top 1000 features
   min_df=2,          # Minimum document frequency
   max_df=0.8,        # Maximum document frequency
   ngram_range=(1, 2), # Unigrams and bigrams
   stop_words='english'
)

tfidf_features = tfidf_vectorizer.fit_transform(social_df['processed_text'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"TF-IDF Features Shape: {tfidf_features.shape}")
print(f"Feature Names (first 10): {list(tfidf_feature_names[:10])}")

# 2. Count Vectorizer Features
print("\n📊 Creating Count Vectorizer features...")
count_vectorizer = CountVectorizer(
   max_features=500,
   min_df=2,
   max_df=0.8,
   ngram_range=(1, 2),
   stop_words='english'
)

count_features = count_vectorizer.fit_transform(social_df['processed_text'])
count_feature_names = count_vectorizer.get_feature_names_out()

print(f"Count Features Shape: {count_features.shape}")

# 3. Sentiment Features using TextBlob
print("\n😊 Extracting sentiment features...")
def extract_sentiment_features(text):
   """Extract sentiment polarity and subjectivity"""
   blob = TextBlob(text)
   return blob.sentiment.polarity, blob.sentiment.subjectivity

sentiment_data = social_df['processed_text'].apply(extract_sentiment_features)
social_df['sentiment_polarity'] = [x[0] for x in sentiment_data]
social_df['sentiment_subjectivity'] = [x[1] for x in sentiment_data]

# 4. VADER Sentiment Analysis
print("😊 Applying VADER sentiment analysis...")
sia = SentimentIntensityAnalyzer()

def get_vader_scores(text):
   """Get VADER sentiment scores"""
   scores = sia.polarity_scores(text)
   return scores['compound'], scores['pos'], scores['neu'], scores['neg']

vader_data = social_df['processed_text'].apply(get_vader_scores)
social_df['vader_compound'] = [x[0] for x in vader_data]
social_df['vader_positive'] = [x[1] for x in vader_data]
social_df['vader_neutral'] = [x[2] for x in vader_data]
social_df['vader_negative'] = [x[3] for x in vader_data]

# 5. Additional linguistic features
print("📝 Creating additional linguistic features...")

def extract_linguistic_features(text):
   """Extract additional linguistic features"""
   words = text.split()
   
   # Average word length
   avg_word_length = np.mean([len(word) for word in words]) if words else 0
   
   # Unique word ratio
   unique_word_ratio = len(set(words)) / len(words) if words else 0
   
   # Punctuation counts
   periods = text.count('.')
   commas = text.count(',')
   
   return avg_word_length, unique_word_ratio, periods, commas

linguistic_data = social_df['processed_text'].apply(extract_linguistic_features)
social_df['avg_word_length'] = [x[0] for x in linguistic_data]
social_df['unique_word_ratio'] = [x[1] for x in linguistic_data]
social_df['period_count'] = [x[2] for x in linguistic_data]
social_df['comma_count'] = [x[3] for x in linguistic_data]

print("✅ Feature extraction completed!")

# Summary of all features
print("\n📋 FEATURE SUMMARY:")
print("=" * 30)
print(f"TF-IDF Features: {tfidf_features.shape[1]}")
print(f"Count Features: {count_features.shape[1]}")
print(f"Numerical Features: {len(numerical_cols)}")
print(f"Sentiment Features: 6 (TextBlob + VADER)")
print(f"Linguistic Features: 4")
print(f"Total Features Available: {tfidf_features.shape[1] + count_features.shape[1] + len(numerical_cols) + 10}")

# Show sentiment analysis examples
print("\n😊 SENTIMENT ANALYSIS EXAMPLES:")
print("=" * 40)
for emotion in ['Happy', 'Sad', 'Angry']:
   sample = social_df[social_df['emotion'] == emotion].iloc[0]
   print(f"\n{emotion.upper()} Example:")
   print(f"Text: \"{sample['post_text'][:100]}...\"")
   print(f"TextBlob Polarity: {sample['sentiment_polarity']:.3f}")
   print(f"VADER Compound: {sample['vader_compound']:.3f}")
   print(f"VADER Positive: {sample['vader_positive']:.3f}")
   print(f"VADER Negative: {sample['vader_negative']:.3f}")

In [ ]:
#8: Sentiment Features Analysis
# Sentiment features analysis
print("📊 SENTIMENT FEATURES ANALYSIS")
print("=" * 50)

# Sentiment distribution by emotion
sentiment_features = ['sentiment_polarity', 'sentiment_subjectivity', 
                     'vader_compound', 'vader_positive', 'vader_neutral', 'vader_negative']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(sentiment_features):
    sns.boxplot(data=social_df, x='emotion', y=feature, palette=colors, ax=axes[i])
    axes[i].set_title(f'📈 {feature.replace("_", " ").title()} by Emotion', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation matrix of sentiment features
print("\n🔗 SENTIMENT FEATURES CORRELATION:")
print("=" * 40)

sentiment_corr = social_df[sentiment_features].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(sentiment_corr, annot=True, cmap='coolwarm', center=0,
           square=True, fmt='.3f')
plt.title('🔗 Sentiment Features Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Sentiment statistics by emotion
print("\n📊 SENTIMENT STATISTICS BY EMOTION:")
print("=" * 45)

sentiment_stats = social_df.groupby('emotion')[sentiment_features].mean()
print(sentiment_stats.round(3))

# Find most discriminative sentiment features
print("\n🎯 MOST DISCRIMINATIVE SENTIMENT FEATURES:")
print("=" * 50)

for feature in sentiment_features:
    emotion_means = social_df.groupby('emotion')[feature].mean()
    max_emotion = emotion_means.idxmax()
    min_emotion = emotion_means.idxmin()
    difference = emotion_means.max() - emotion_means.min()
    
    print(f"{feature.replace('_', ' ').title()}:")
    print(f"  Highest: {max_emotion} ({emotion_means.max():.3f})")
    print(f"  Lowest: {min_emotion} ({emotion_means.min():.3f})")
    print(f"  Range: {difference:.3f}")
    print()

# Scatter plots of key sentiment features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# TextBlob polarity vs subjectivity
for i, emotion in enumerate(emotions):
    emotion_data = social_df[social_df['emotion'] == emotion]
    axes[0].scatter(emotion_data['sentiment_polarity'], emotion_data['sentiment_subjectivity'],
                   alpha=0.6, label=emotion, color=colors[i], s=30)

axes[0].set_xlabel('Sentiment Polarity')
axes[0].set_ylabel('Sentiment Subjectivity')
axes[0].set_title('📊 TextBlob: Polarity vs Subjectivity', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# VADER positive vs negative
for i, emotion in enumerate(emotions):
    emotion_data = social_df[social_df['emotion'] == emotion]
    axes[1].scatter(emotion_data['vader_positive'], emotion_data['vader_negative'],
                   alpha=0.6, label=emotion, color=colors[i], s=30)

axes[1].set_xlabel('VADER Positive Score')
axes[1].set_ylabel('VADER Negative Score')
axes[1].set_title('📊 VADER: Positive vs Negative Scores', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#9: Data Preprocessing for Machine Learning
print("🔧 DATA PREPROCESSING FOR MACHINE LEARNING")
print("=" * 60)

# Combine all numerical features
print("📊 Combining all numerical features...")

# Original numerical features
original_numerical = ['text_length', 'word_count', 'emoji_count', 'hashtag_count', 
                     'mention_count', 'exclamation_count', 'question_count', 'caps_ratio',
                     'hour', 'day_of_week']

# Sentiment features
sentiment_numerical = ['sentiment_polarity', 'sentiment_subjectivity', 
                      'vader_compound', 'vader_positive', 'vader_neutral', 'vader_negative']

# Linguistic features
linguistic_numerical = ['avg_word_length', 'unique_word_ratio', 'period_count', 'comma_count']

# Combine all numerical features
all_numerical_features = original_numerical + sentiment_numerical + linguistic_numerical

print(f"Total Numerical Features: {len(all_numerical_features)}")

# Encode categorical features
print("\n🏷️ Encoding categorical features...")
label_encoders = {}
categorical_features = ['platform']

social_df_processed = social_df.copy()

for feature in categorical_features:
    le = LabelEncoder()
    social_df_processed[feature + '_encoded'] = le.fit_transform(social_df_processed[feature])
    label_encoders[feature] = le
    all_numerical_features.append(feature + '_encoded')
    print(f"  {feature}: {list(le.classes_)}")

# Create time-based features
print("\n⏰ Creating time-based features...")
social_df_processed['is_weekend'] = (social_df_processed['day_of_week'] >= 5).astype(int)
social_df_processed['is_night'] = ((social_df_processed['hour'] >= 22) | (social_df_processed['hour'] <= 5)).astype(int)
social_df_processed['is_peak_hours'] = social_df_processed['hour'].isin([12, 13, 18, 19, 20, 21]).astype(int)

time_features = ['is_weekend', 'is_night', 'is_peak_hours']
all_numerical_features.extend(time_features)

# Feature selection and preparation
print("\n🎯 Preparing features for modeling...")

# Option 1: Use TF-IDF + Numerical Features
print("\n📊 Option 1: TF-IDF + Numerical Features")
X_numerical = social_df_processed[all_numerical_features]
X_tfidf_dense = tfidf_features.toarray()

# Combine TF-IDF and numerical features
X_combined = np.hstack([X_tfidf_dense, X_numerical])
feature_names_combined = list(tfidf_feature_names) + all_numerical_features

print(f"Combined Features Shape: {X_combined.shape}")
print(f"TF-IDF Features: {X_tfidf_dense.shape[1]}")
print(f"Numerical Features: {X_numerical.shape[1]}")

# Option 2: Use only numerical features (for simpler interpretation)
print("\n📊 Option 2: Numerical Features Only")
X_numerical_only = X_numerical.values

print(f"Numerical Features Shape: {X_numerical_only.shape}")

# Prepare target variable
print("\n🎯 Preparing target variable...")
y = social_df_processed['emotion']
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print(f"Target Classes: {list(target_encoder.classes_)}")
print(f"Encoded as: {list(range(len(target_encoder.classes_)))}")

# Split the data for both feature sets
print("\n✂️ Splitting data...")

# For combined features (TF-IDF + Numerical)
X_train_combined, X_test_combined, y_train, y_test = train_test_split(
    X_combined, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# For numerical features only
X_train_numerical, X_test_numerical, _, _ = train_test_split(
    X_numerical_only, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training Set: {X_train_combined.shape[0]} samples")
print(f"Test Set: {X_test_combined.shape[0]} samples")

# Feature scaling
print("\n📏 Scaling features...")

# Scale combined features
scaler_combined = StandardScaler()
X_train_combined_scaled = scaler_combined.fit_transform(X_train_combined)
X_test_combined_scaled = scaler_combined.transform(X_test_combined)

# Scale numerical features only
scaler_numerical = StandardScaler()
X_train_numerical_scaled = scaler_numerical.fit_transform(X_train_numerical)
X_test_numerical_scaled = scaler_numerical.transform(X_test_numerical)

# Save preprocessing objects
preprocessing_objects = {
    'tfidf_vectorizer': tfidf_vectorizer,
    'label_encoders': label_encoders,
    'target_encoder': target_encoder,
    'scaler_combined': scaler_combined,
    'scaler_numerical': scaler_numerical,
    'feature_names_combined': feature_names_combined,
    'all_numerical_features': all_numerical_features
}

print("\n✅ PREPROCESSING COMPLETE!")
print("=" * 30)
print(f"✓ Combined features ready: {X_train_combined_scaled.shape}")
print(f"✓ Numerical features ready: {X_train_numerical_scaled.shape}")
print(f"✓ Target encoded: {len(target_encoder.classes_)} classes")
print(f"✓ Data split: {len(X_train_combined)} train, {len(X_test_combined)} test")
print(f"✓ Features scaled and ready for modeling")

# Display feature importance preview
print(f"\n📋 FEATURE GROUPS SUMMARY:")
print("=" * 35)
print(f"📝 Text Features (TF-IDF): {len(tfidf_feature_names)}")
print(f"📊 Original Numerical: {len(original_numerical)}")
print(f"😊 Sentiment Features: {len(sentiment_numerical)}")
print(f"🔤 Linguistic Features: {len(linguistic_numerical)}")
print(f"🏷️ Categorical Encoded: {len(categorical_features)}")
print(f"⏰ Time Features: {len(time_features)}")
print(f"📈 Total Features: {len(feature_names_combined)}")

In [ ]:
#10: Custom KNN Implementation for Text
import numpy as np
import time
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

class CustomTextKNN:
    def __init__(self, k=5, distance_metric='cosine', weights='uniform'):
        """
        Custom K-Nearest Neighbors for text classification
        
        Parameters:
        - k: number of neighbors
        - distance_metric: 'cosine', 'euclidean', 'manhattan'
        - weights: 'uniform' or 'distance'
        """
        self.k = k
        self.distance_metric = distance_metric
        self.weights = weights
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y):
        """Store training data"""
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        
    def _calculate_distance(self, x1, x2):
        """Calculate distance between two points"""
        if self.distance_metric == 'cosine':
            # Cosine similarity (convert to distance)
            similarity = np.dot(x1, x2) / (np.linalg.norm(x1) * np.linalg.norm(x2))
            return 1 - similarity
        elif self.distance_metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        elif self.distance_metric == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        else:
            raise ValueError("Unknown distance metric")
    
    def _get_neighbors(self, test_point):
        """Get k nearest neighbors for a test point"""
        distances = []
        
        for i, train_point in enumerate(self.X_train):
            dist = self._calculate_distance(test_point, train_point)
            distances.append((dist, self.y_train[i], i))
        
        # Sort by distance and get k nearest
        distances.sort(key=lambda x: x[0])
        return distances[:self.k]
    
    def predict(self, X_test):
        """Make predictions for test data"""
        predictions = []
        
        for test_point in X_test:
            neighbors = self._get_neighbors(test_point)
            
            if self.weights == 'uniform':
                # Simple majority vote
                neighbor_labels = [neighbor[1] for neighbor in neighbors]
                prediction = Counter(neighbor_labels).most_common(1)[0][0]
            else:  # distance weighting
                # Weighted vote based on inverse distance
                weighted_votes = {}
                for distance, label, _ in neighbors:
                    weight = 1 / (distance + 1e-8)  # Add small value to avoid division by zero
                    if label in weighted_votes:
                        weighted_votes[label] += weight
                    else:
                        weighted_votes[label] = weight
                
                prediction = max(weighted_votes.items(), key=lambda x: x[1])[0]
            
            predictions.append(prediction)
            
        return np.array(predictions)
    
    def predict_proba(self, X_test):
        """Predict class probabilities"""
        probabilities = []
        
        for test_point in X_test:
            neighbors = self._get_neighbors(test_point)
            
            if self.weights == 'uniform':
                neighbor_labels = [neighbor[1] for neighbor in neighbors]
                label_counts = Counter(neighbor_labels)
                total_neighbors = len(neighbors)
                
                prob_dict = {}
                for label, count in label_counts.items():
                    prob_dict[label] = count / total_neighbors
            else:
                # Distance-weighted probabilities
                weighted_votes = {}
                total_weight = 0
                
                for distance, label, _ in neighbors:
                    weight = 1 / (distance + 1e-8)
                    total_weight += weight
                    if label in weighted_votes:
                        weighted_votes[label] += weight
                    else:
                        weighted_votes[label] = weight
                
                prob_dict = {}
                for label, weight in weighted_votes.items():
                    prob_dict[label] = weight / total_weight
            
            probabilities.append(prob_dict)
            
        return probabilities

# Test custom KNN implementation
print("🧮 TESTING CUSTOM TEXT KNN IMPLEMENTATION")
print("=" * 60)

# Test with different parameters
k_values = [3, 5, 7, 9]
distance_metrics = ['cosine', 'euclidean', 'manhattan']
weight_options = ['uniform', 'distance']

custom_results = {}

print("🧪 Testing Custom KNN with different parameters...")
print("-" * 80)
print(f"{'K':<5} {'Metric':<12} {'Weight':<10} {'Accuracy':<10} {'Time':<8}")
print("-" * 80)

for k in k_values:
    for metric in distance_metrics:
        for weight in weight_options:
            start_time = time.time()
            
            # Use numerical features only for faster computation
            custom_knn = CustomTextKNN(k=k, distance_metric=metric, weights=weight)
            custom_knn.fit(X_train_numerical_scaled, y_train)
            
            # Make predictions on a subset for speed
            test_subset = X_test_numerical_scaled[:100]  # First 100 samples
            y_test_subset = y_test[:100]
            
            y_pred = custom_knn.predict(test_subset)
            accuracy = np.mean(y_pred == y_test_subset)
            
            training_time = time.time() - start_time
            
            custom_results[f'k{k}_{metric}_{weight}'] = accuracy
            
            print(f"{k:<5} {metric:<12} {weight:<10} {accuracy:.4f}    {training_time:.3f}s")

# Find best custom configuration
best_custom = max(custom_results.items(), key=lambda x: x[1])
print(f"\n🏆 Best Custom KNN: {best_custom[0]} with accuracy {best_custom[1]:.4f}")

# Test with text features (TF-IDF) - use smaller subset due to computational cost
print(f"\n📝 Testing with TF-IDF features (subset)...")
custom_text_knn = CustomTextKNN(k=5, distance_metric='cosine', weights='distance')
custom_text_knn.fit(X_train_combined_scaled[:500], y_train[:500])  # Use subset for speed

# Test on small subset
test_subset_combined = X_test_combined_scaled[:50]
y_test_subset_combined = y_test[:50]

y_pred_text = custom_text_knn.predict(test_subset_combined)
accuracy_text = np.mean(y_pred_text == y_test_subset_combined)

print(f"Custom KNN with TF-IDF features accuracy: {accuracy_text:.4f}")

# Example predictions with probabilities
print(f"\n🔮 EXAMPLE PREDICTIONS WITH PROBABILITIES:")
print("=" * 50)

# Use the best numerical model for examples
best_params = best_custom[0].split('_')
best_k = int(best_params[0][1:])
best_metric = best_params[1]
best_weight = best_params[2]

example_knn = CustomTextKNN(k=best_k, distance_metric=best_metric, weights=best_weight)
example_knn.fit(X_train_numerical_scaled, y_train)

# Get probabilities for first 3 test samples
example_probabilities = example_knn.predict_proba(X_test_numerical_scaled[:3])
example_predictions = example_knn.predict(X_test_numerical_scaled[:3])

for i in range(3):
    actual_emotion = target_encoder.inverse_transform([y_test[i]])[0]
    predicted_emotion = target_encoder.inverse_transform([example_predictions[i]])[0]
    
    print(f"\nSample {i+1}:")
    print(f"  Text: \"{social_df.iloc[X_test_combined.shape[0] - len(X_test_combined) + i]['post_text'][:80]}...\"")
    print(f"  Predicted: {predicted_emotion}")
    print(f"  Actual: {actual_emotion}")
    print(f"  Correct: {'✅' if predicted_emotion == actual_emotion else '❌'}")
    print(f"  Probabilities:")
    
    for class_label, prob in example_probabilities[i].items():
        class_name = target_encoder.inverse_transform([class_label])[0]
        bar = "█" * int(prob * 20)
        print(f"    {class_name:<8}: {prob:.3f} {bar}")

print(f"\n💡 CUSTOM KNN INSIGHTS:")
print("=" * 30)
print(f"• Cosine distance works well for text features")
print(f"• Distance weighting can improve performance")
print(f"• Higher K values may smooth out noise")
print(f"• Text features (TF-IDF) capture semantic meaning")
print(f"• Numerical features capture posting behavior")

In [ ]:
#11: Scikit-learn KNN Implementation and Optimization
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import time

print("🔬 SCIKIT-LEARN KNN IMPLEMENTATION AND OPTIMIZATION")
print("=" * 70)

# Define parameter grid for comprehensive search
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'cosine', 'minkowski'],
    'p': [1, 2]  # Only used when metric='minkowski'
}

# Filter param grid for minkowski metric
def filter_params(params):
    """Filter parameters to avoid invalid combinations"""
    if params['metric'] != 'minkowski':
        return {k: v for k, v in params.items() if k != 'p'}
    return params

# Manual parameter testing for better control
k_values = [3, 5, 7, 9, 11, 15]
weights = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan', 'cosine']

sklearn_results = []

print("🧪 Testing Scikit-learn KNN with different parameters...")
print("-" * 90)
print(f"{'Feature Set':<15} {'K':<5} {'Weight':<10} {'Metric':<12} {'CV Score':<10} {'Test Acc':<10} {'Time':<8}")
print("-" * 90)

# Test with numerical features only
print("Testing with Numerical Features Only:")
for k in k_values:
    for weight in weights:
        for metric in metrics:
            start_time = time.time()
            
            knn = KNeighborsClassifier(n_neighbors=k, weights=weight, metric=metric)
            
            # Cross-validation
            cv_scores = cross_val_score(knn, X_train_numerical_scaled, y_train, 
                                      cv=5, scoring='accuracy')
            cv_mean = cv_scores.mean()
            
            # Train and test
            knn.fit(X_train_numerical_scaled, y_train)
            test_accuracy = knn.score(X_test_numerical_scaled, y_test)
            
            training_time = time.time() - start_time
            
            sklearn_results.append({
                'feature_set': 'numerical',
                'k': k,
                'weight': weight,
                'metric': metric,
                'cv_score': cv_mean,
                'test_accuracy': test_accuracy,
                'time': training_time
            })
            
            print(f"{'Numerical':<15} {k:<5} {weight:<10} {metric:<12} {cv_mean:.4f}    {test_accuracy:.4f}    {training_time:.3f}s")

# Test with combined features (subset due to computational cost)
print(f"\nTesting with Combined Features (TF-IDF + Numerical):")
k_values_combined = [3, 5, 7]  # Reduced for computational efficiency
metrics_combined = ['euclidean', 'cosine']

for k in k_values_combined:
    for weight in weights:
        for metric in metrics_combined:
            start_time = time.time()
            
            knn = KNeighborsClassifier(n_neighbors=k, weights=weight, metric=metric)
            
            # Use subset for cross-validation due to computational cost
            subset_size = 1000
            X_subset = X_train_combined_scaled[:subset_size]
            y_subset = y_train[:subset_size]
            
            cv_scores = cross_val_score(knn, X_subset, y_subset, cv=3, scoring='accuracy')
            cv_mean = cv_scores.mean()
            
            # Train on full training set and test
            knn.fit(X_train_combined_scaled, y_train)
            test_accuracy = knn.score(X_test_combined_scaled, y_test)
            
            training_time = time.time() - start_time
            
            sklearn_results.append({
                'feature_set': 'combined',
                'k': k,
                'weight': weight,
                'metric': metric,
                'cv_score': cv_mean,
                'test_accuracy': test_accuracy,
                'time': training_time
            })
            
            print(f"{'Combined':<15} {k:<5} {weight:<10} {metric:<12} {cv_mean:.4f}    {test_accuracy:.4f}    {training_time:.3f}s")

# Find best models for each feature set
numerical_results = [r for r in sklearn_results if r['feature_set'] == 'numerical']
combined_results = [r for r in sklearn_results if r['feature_set'] == 'combined']

best_numerical = max(numerical_results, key=lambda x: x['test_accuracy'])
best_combined = max(combined_results, key=lambda x: x['test_accuracy'])

print("\n" + "=" * 70)
print("🏆 BEST MODELS COMPARISON")
print("=" * 70)

print("📊 Best Numerical Features Model:")
print(f"  K: {best_numerical['k']}")
print(f"  Weight: {best_numerical['weight']}")
print(f"  Metric: {best_numerical['metric']}")
print(f"  CV Score: {best_numerical['cv_score']:.4f}")
print(f"  Test Accuracy: {best_numerical['test_accuracy']:.4f}")
print(f"  Training Time: {best_numerical['time']:.3f}s")

print(f"\n📝 Best Combined Features Model:")
print(f"  K: {best_combined['k']}")
print(f"  Weight: {best_combined['weight']}")
print(f"  Metric: {best_combined['metric']}")
print(f"  CV Score: {best_combined['cv_score']:.4f}")
print(f"  Test Accuracy: {best_combined['test_accuracy']:.4f}")
print(f"  Training Time: {best_combined['time']:.3f}s")

# Train final models with best parameters
print(f"\n🎯 Training Final Models...")

# Best numerical model
final_numerical_knn = KNeighborsClassifier(
   n_neighbors=best_numerical['k'],
   weights=best_numerical['weight'],
   metric=best_numerical['metric']
)
final_numerical_knn.fit(X_train_numerical_scaled, y_train)

# Best combined model
final_combined_knn = KNeighborsClassifier(
   n_neighbors=best_combined['k'],
   weights=best_combined['weight'],
   metric=best_combined['metric']
)
final_combined_knn.fit(X_train_combined_scaled, y_train)

# Get predictions
y_pred_numerical = final_numerical_knn.predict(X_test_numerical_scaled)
y_pred_combined = final_combined_knn.predict(X_test_combined_scaled)

final_accuracy_numerical = accuracy_score(y_test, y_pred_numerical)
final_accuracy_combined = accuracy_score(y_test, y_pred_combined)

print(f"Final Numerical Model Accuracy: {final_accuracy_numerical:.4f}")
print(f"Final Combined Model Accuracy: {final_accuracy_combined:.4f}")

# Performance comparison visualization
comparison_data = {
   'Model': ['Numerical Features', 'Combined Features'],
   'Test Accuracy': [final_accuracy_numerical, final_accuracy_combined],
   'CV Score': [best_numerical['cv_score'], best_combined['cv_score']],
   'Training Time': [best_numerical['time'], best_combined['time']],
   'Features Count': [len(all_numerical_features), len(feature_names_combined)]
}

comparison_df = pd.DataFrame(comparison_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Accuracy comparison
bars1 = axes[0].bar(comparison_df['Model'], comparison_df['Test Accuracy'], 
                  color=['#4ECDC4', '#FF6B6B'], alpha=0.8)
axes[0].set_title('🎯 Test Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)

for bar in bars1:
   height = bar.get_height()
   axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
               f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

# Feature count comparison
bars2 = axes[1].bar(comparison_df['Model'], comparison_df['Features Count'], 
                  color=['#96CEB4', '#FFEAA7'], alpha=0.8)
axes[1].set_title('📊 Feature Count Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Features')
axes[1].tick_params(axis='x', rotation=45)

for bar in bars2:
   height = bar.get_height()
   axes[1].text(bar.get_x() + bar.get_width()/2., height + 20,
               f'{int(height)}', ha='center', va='bottom', fontweight='bold')

# Training time comparison
bars3 = axes[2].bar(comparison_df['Model'], comparison_df['Training Time'], 
                  color=['#45B7D1', '#D63031'], alpha=0.8)
axes[2].set_title('⏱️ Training Time Comparison', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Time (seconds)')
axes[2].tick_params(axis='x', rotation=45)

for bar in bars3:
   height = bar.get_height()
   axes[2].text(bar.get_x() + bar.get_width()/2., height + 0.1,
               f'{height:.2f}s', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Detailed parameter analysis
print(f"\n📈 PARAMETER ANALYSIS:")
print("=" * 40)

# K value analysis
k_performance = {}
for result in sklearn_results:
   k = result['k']
   if k not in k_performance:
       k_performance[k] = []
   k_performance[k].append(result['test_accuracy'])

print(f"Average accuracy by K value:")
for k in sorted(k_performance.keys()):
   avg_acc = np.mean(k_performance[k])
   print(f"  K={k}: {avg_acc:.4f} (±{np.std(k_performance[k]):.4f})")

# Weight analysis
weight_performance = {'uniform': [], 'distance': []}
for result in sklearn_results:
   weight_performance[result['weight']].append(result['test_accuracy'])

print(f"\nAverage accuracy by weight method:")
for weight in weight_performance:
   avg_acc = np.mean(weight_performance[weight])
   print(f"  {weight}: {avg_acc:.4f} (±{np.std(weight_performance[weight]):.4f})")

# Metric analysis
metric_performance = {}
for result in sklearn_results:
   metric = result['metric']
   if metric not in metric_performance:
       metric_performance[metric] = []
   metric_performance[metric].append(result['test_accuracy'])

print(f"\nAverage accuracy by distance metric:")
for metric in sorted(metric_performance.keys()):
   avg_acc = np.mean(metric_performance[metric])
   print(f"  {metric}: {avg_acc:.4f} (±{np.std(metric_performance[metric]):.4f})")

# Choose final model based on best performance
if final_accuracy_combined > final_accuracy_numerical:
   final_model = final_combined_knn
   final_accuracy = final_accuracy_combined
   final_X_test = X_test_combined_scaled
   final_feature_set = "Combined (TF-IDF + Numerical)"
   print(f"\n✅ Selected Final Model: Combined Features KNN")
else:
   final_model = final_numerical_knn
   final_accuracy = final_accuracy_numerical
   final_X_test = X_test_numerical_scaled
   final_feature_set = "Numerical Only"
   print(f"\n✅ Selected Final Model: Numerical Features KNN")

print(f"   Accuracy: {final_accuracy:.4f}")
print(f"   Feature Set: {final_feature_set}")

In [ ]:
#12: Model Evaluation and Analysis
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("📊 COMPREHENSIVE MODEL EVALUATION AND ANALYSIS")
print("=" * 70)

# Use the best performing model for detailed evaluation
best_model = final_combined_knn if final_accuracy_combined > final_accuracy_numerical else final_numerical_knn
best_X_test = X_test_combined_scaled if final_accuracy_combined > final_accuracy_numerical else X_test_numerical_scaled
best_predictions = y_pred_combined if final_accuracy_combined > final_accuracy_numerical else y_pred_numerical

# Classification Report
print("📈 CLASSIFICATION REPORT")
print("=" * 40)
class_names = target_encoder.classes_
print(classification_report(y_test, best_predictions, target_names=class_names))

# Confusion Matrix
print(f"\n🔍 CONFUSION MATRIX")
print("=" * 30)
cm = confusion_matrix(y_test, best_predictions)
print(cm)

# Visualize Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('🔍 Confusion Matrix - KNN Emotion Classification', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Emotion', fontsize=12)
plt.ylabel('True Emotion', fontsize=12)
plt.tight_layout()
plt.show()

# Detailed metrics per class
print(f"\n📊 DETAILED METRICS PER EMOTION")
print("=" * 50)
precision, recall, fscore, support = precision_recall_fscore_support(y_test, best_predictions)

metrics_df = pd.DataFrame({
    'Emotion': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': fscore,
    'Support': support
})

print(metrics_df.round(4))

# Visualize per-class performance
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Precision
bars1 = axes[0].bar(class_names, precision, color=colors, alpha=0.8)
axes[0].set_title('🎯 Precision by Emotion', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Precision')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)

for i, bar in enumerate(bars1):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

# Recall
bars2 = axes[1].bar(class_names, recall, color=colors, alpha=0.8)
axes[1].set_title('🔄 Recall by Emotion', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Recall')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=45)

for i, bar in enumerate(bars2):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

# F1-Score
bars3 = axes[2].bar(class_names, fscore, color=colors, alpha=0.8)
axes[2].set_title('⚖️ F1-Score by Emotion', fontsize=14, fontweight='bold')
axes[2].set_ylabel('F1-Score')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=45)

for i, bar in enumerate(bars3):
    height = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Overall performance metrics
print(f"\n🎯 OVERALL PERFORMANCE SUMMARY")
print("=" * 40)
print(f"Accuracy: {final_accuracy:.4f}")
print(f"Macro Avg Precision: {precision.mean():.4f}")
print(f"Macro Avg Recall: {recall.mean():.4f}")
print(f"Macro Avg F1-Score: {fscore.mean():.4f}")
print(f"Weighted Avg F1-Score: {np.average(fscore, weights=support):.4f}")

# Error analysis
print(f"\n❌ ERROR ANALYSIS")
print("=" * 25)

# Find misclassified samples
misclassified_indices = np.where(y_test != best_predictions)[0]
print(f"Total misclassified: {len(misclassified_indices)} out of {len(y_test)} ({len(misclassified_indices)/len(y_test)*100:.1f}%)")

# Most confused emotion pairs
confusion_pairs = {}
for i in misclassified_indices:
    true_emotion = target_encoder.inverse_transform([y_test[i]])[0]
    pred_emotion = target_encoder.inverse_transform([best_predictions[i]])[0]
    pair = f"{true_emotion} → {pred_emotion}"
    confusion_pairs[pair] = confusion_pairs.get(pair, 0) + 1

print(f"\nMost common misclassifications:")
sorted_pairs = sorted(confusion_pairs.items(), key=lambda x: x[1], reverse=True)
for pair, count in sorted_pairs[:5]:
    print(f"  {pair}: {count} times")

# Show examples of misclassified posts
print(f"\n📱 MISCLASSIFIED EXAMPLES:")
print("=" * 35)

shown_examples = 0
for idx in misclassified_indices[:5]:  # Show first 5 misclassified examples
    original_idx = len(social_df) - len(y_test) + idx  # Map back to original dataframe
    if original_idx < len(social_df):
        post_text = social_df.iloc[original_idx]['post_text']
        true_emotion = target_encoder.inverse_transform([y_test[idx]])[0]
        pred_emotion = target_encoder.inverse_transform([best_predictions[idx]])[0]
        
        print(f"\nExample {shown_examples + 1}:")
        print(f"  Text: \"{post_text[:100]}...\"")
        print(f"  True: {true_emotion}")
        print(f"  Predicted: {pred_emotion}")
        
        shown_examples += 1

# Feature importance analysis (for numerical features)
if hasattr(best_model, 'feature_importances_') or True:  # KNN doesn't have feature_importances_, so use permutation
    print(f"\n🔍 FEATURE IMPORTANCE ANALYSIS")
    print("=" * 40)
    
    from sklearn.inspection import permutation_importance
    
    # Calculate permutation importance
    if final_accuracy_combined > final_accuracy_numerical:
        # For combined features, analyze numerical features separately
        print("Analyzing numerical features importance in combined model...")
        numerical_start_idx = len(tfidf_feature_names)
        X_test_numerical_part = best_X_test[:, numerical_start_idx:]
        
        perm_importance = permutation_importance(
            best_model, best_X_test, y_test, 
            n_repeats=5, random_state=42, scoring='accuracy'
        )
        
        # Get top features from numerical part
        numerical_importance = perm_importance.importances_mean[numerical_start_idx:]
        feature_importance_df = pd.DataFrame({
            'feature': all_numerical_features,
            'importance': numerical_importance
        }).sort_values('importance', ascending=False)
        
    else:
        # For numerical only model
        perm_importance = permutation_importance(
            best_model, best_X_test, y_test, 
            n_repeats=5, random_state=42, scoring='accuracy'
        )
        
        feature_importance_df = pd.DataFrame({
            'feature': all_numerical_features,
            'importance': perm_importance.importances_mean
        }).sort_values('importance', ascending=False)
    
    print("Top 10 Most Important Numerical Features:")
    print(feature_importance_df.head(10))
    
    # Visualize feature importance
    plt.figure(figsize=(12, 8))
    top_features = feature_importance_df.head(10)
    bars = plt.barh(range(len(top_features)), top_features['importance'], 
                   color='steelblue', alpha=0.8)
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Permutation Importance')
    plt.title('🔍 Top 10 Feature Importance (Numerical Features)', 
              fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Model complexity analysis
print(f"\n🧠 MODEL COMPLEXITY ANALYSIS")
print("=" * 35)
print(f"Model Type: K-Nearest Neighbors")
print(f"K Value: {best_model.n_neighbors}")
print(f"Distance Metric: {best_model.metric}")
print(f"Weighting: {best_model.weights}")
print(f"Feature Dimensions: {best_X_test.shape[1]}")
print(f"Training Samples: {len(y_train)}")
print(f"Memory Usage: ~{best_X_test.nbytes / 1024 / 1024:.1f} MB for test data")

In [ ]:
#13: Interactive Emotion Prediction Demo
import ipywidgets as widgets
from IPython.display import display, clear_output
import re

print("🎛️ INTERACTIVE EMOTION PREDICTION DEMO")
print("=" * 60)

# Create input widgets
text_input = widgets.Textarea(
    value="I'm so excited about my vacation! This is going to be amazing! 🎉",
    placeholder="Enter your social media post here...",
    description="Your Post:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', height='100px')
)

platform_dropdown = widgets.Dropdown(
    options=['Twitter', 'Instagram', 'Facebook'],
    value='Twitter',
    description='Platform:'
)

hour_slider = widgets.IntSlider(
    value=12,
    min=0,
    max=23,
    description='Hour:',
    style={'description_width': 'initial'}
)

day_dropdown = widgets.Dropdown(
    options=[(day, i) for i, day in enumerate(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])],
    value=0,
    description='Day of Week:'
)

predict_button = widgets.Button(
    description='🔮 Predict Emotion',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

output_area = widgets.Output()

def predict_emotion(button):
    with output_area:
        clear_output(wait=True)
        
        try:
            # Get input text
            post_text = text_input.value.strip()
            
            if not post_text:
                print("❌ Please enter some text to analyze!")
                return
            
            # Extract features from text
            processed_text = preprocess_text(post_text)
            
            # Text features
            text_length = len(post_text)
            word_count = len(post_text.split())
            emoji_count = len(re.findall(r'[😀-🿿]', post_text))
            hashtag_count = post_text.count('#')
            mention_count = post_text.count('@')
            exclamation_count = post_text.count('!')
            question_count = post_text.count('?')
            caps_ratio = sum(1 for c in post_text if c.isupper()) / len(post_text) if len(post_text) > 0 else 0
            
            # Sentiment features
            blob = TextBlob(processed_text)
            sentiment_polarity = blob.sentiment.polarity
            sentiment_subjectivity = blob.sentiment.subjectivity
            
            # VADER sentiment
            sia = SentimentIntensityAnalyzer()
            vader_scores = sia.polarity_scores(processed_text)
            vader_compound = vader_scores['compound']
            vader_positive = vader_scores['pos']
            vader_neutral = vader_scores['neu']
            vader_negative = vader_scores['neg']
            
            # Linguistic features
            words = processed_text.split()
            avg_word_length = np.mean([len(word) for word in words]) if words else 0
            unique_word_ratio = len(set(words)) / len(words) if words else 0
            period_count = processed_text.count('.')
            comma_count = processed_text.count(',')
            
            # Time and platform features
            hour = hour_slider.value
            day_of_week = day_dropdown.value
            platform_encoded = label_encoders['platform'].transform([platform_dropdown.value])[0]
            
            # Additional features
            is_weekend = int(day_of_week >= 5)
            is_night = int(hour >= 22 or hour <= 5)
            is_peak_hours = int(hour in [12, 13, 18, 19, 20, 21])
            
            # Prepare feature vector
            if final_accuracy_combined > final_accuracy_numerical:
                # Use combined model
                # First, get TF-IDF features
                tfidf_vector = tfidf_vectorizer.transform([processed_text]).toarray()
                
                # Then, prepare numerical features
                numerical_features = np.array([[
                    text_length, word_count, emoji_count, hashtag_count,
                    mention_count, exclamation_count, question_count, caps_ratio,
                    hour, day_of_week, sentiment_polarity, sentiment_subjectivity,
                    vader_compound, vader_positive, vader_neutral, vader_negative,
                    avg_word_length, unique_word_ratio, period_count, comma_count,
                    platform_encoded, is_weekend, is_night, is_peak_hours
                ]])
                
                # Combine features
                feature_vector = np.hstack([tfidf_vector, numerical_features])
                feature_vector_scaled = scaler_combined.transform(feature_vector)
                model_to_use = final_combined_knn
                
            else:
                # Use numerical model
                feature_vector = np.array([[
                    text_length, word_count, emoji_count, hashtag_count,
                    mention_count, exclamation_count, question_count, caps_ratio,
                    hour, day_of_week, sentiment_polarity, sentiment_subjectivity,
                    vader_compound, vader_positive, vader_neutral, vader_negative,
                    avg_word_length, unique_word_ratio, period_count, comma_count,
                    platform_encoded, is_weekend, is_night, is_peak_hours
                ]])
                feature_vector_scaled = scaler_numerical.transform(feature_vector)
                model_to_use = final_numerical_knn
            
            # Make prediction
            prediction = model_to_use.predict(feature_vector_scaled)[0]
            probabilities = model_to_use.predict_proba(feature_vector_scaled)[0]
            
            # Get emotion name
            predicted_emotion = target_encoder.inverse_transform([prediction])[0]
            
            # Display results
            print("🎭 EMOTION PREDICTION RESULTS")
            print("=" * 50)
            print(f"📝 Your Post: \"{post_text}\"")
            print(f"📱 Platform: {platform_dropdown.value}")
            print(f"⏰ Time: {hour_slider.value}:00 on {day_dropdown.label}")
            
            print(f"\n🎯 PREDICTED EMOTION: {predicted_emotion.upper()}")
            confidence = probabilities[prediction]
            print(f"🔢 Confidence: {confidence:.1%}")
            
            print(f"\n📊 All Emotion Probabilities:")
            emotion_probs = []
            for i, class_name in enumerate(target_encoder.classes_):
                prob = probabilities[i]
                emotion_probs.append((class_name, prob))
                bar = "█" * int(prob * 30)
                print(f"  {class_name:<10}: {prob:.1%} {bar}")
            
            # Emotion-specific advice
            print(f"\n💡 EMOTION INSIGHTS:")
            print("=" * 25)
            
            emotion_insights = {
                'Happy': "😊 Positive vibes detected! Your post radiates happiness and joy.",
                'Sad': "😢 This post seems to express sadness. Consider reaching out to friends for support.",
                'Angry': "😡 Strong negative emotions detected. Take a moment to cool down before posting.",
                'Fear': "😰 This post shows signs of worry or anxiety. Consider talking to someone you trust.",
                'Surprise': "😲 Wow! This post expresses surprise or amazement about something unexpected.",
                'Neutral': "😐 This appears to be a neutral, informational post without strong emotions."
            }
            
            print(emotion_insights.get(predicted_emotion, "🤔 Interesting emotional expression detected."))
            
            # Feature analysis
            print(f"\n🔍 TEXT ANALYSIS:")
            print("=" * 20)
            print(f"📏 Length: {text_length} characters")
            print(f"📝 Words: {word_count}")
            print(f"😊 Emojis: {emoji_count}")
            print(f"❗ Exclamations: {exclamation_count}")
            print(f"🔠 Caps ratio: {caps_ratio:.1%}")
            print(f"😊 Sentiment polarity: {sentiment_polarity:.2f} (-1=negative, +1=positive)")
            print(f"🎭 VADER compound: {vader_compound:.2f} (-1=negative, +1=positive)")
            
            # Suggestions
            print(f"\n💭 SUGGESTIONS:")
            print("=" * 15)
            if predicted_emotion == 'Angry':
                print("• Consider rephrasing to sound less aggressive")
                print("• Take a break before posting when upset")
                print("• Use more neutral language")
            elif predicted_emotion == 'Sad':
                print("• Consider adding hopeful elements")
                print("• Reach out to friends for support")
                print("• Share positive memories or activities")
            elif predicted_emotion == 'Happy':
                print("• Great positive energy! This will spread joy")
                print("• Consider adding relevant hashtags")
                print("• Share what made you happy")
            elif predicted_emotion == 'Fear':
                print("• Try to reframe anxious thoughts positively")
                print("• Share your concerns with trusted friends")
                print("• Focus on solutions rather than problems")
            elif predicted_emotion == 'Surprise':
                print("• Share more details about what surprised you")
                print("• Your enthusiasm is contagious!")
                print("• Consider adding relevant context")
            else:  # Neutral
                print("• Consider adding personal opinion or emotion")
                print("• Engage your audience with questions")
                print("• Add emojis to make it more expressive")
                
        except Exception as e:
            print(f"❌ Error in prediction: {str(e)}")
            print("Please check your input and try again.")

predict_button.on_click(predict_emotion)

# Create interface
print("🎛️ Adjust the parameters below and enter your post to get emotion prediction:")

interface = widgets.VBox([
    widgets.HTML("<h3>📝 Enter Your Social Media Post</h3>"),
    text_input,
    widgets.HTML("<h3>📊 Post Context</h3>"),
    widgets.HBox([platform_dropdown, hour_slider]),
    day_dropdown,
    widgets.HTML("<br>"),
    predict_button,
    output_area
])

display(interface)

# Pre-fill with sample posts
sample_posts = [
    "I'm so excited about my vacation! This is going to be amazing! 🎉",
    "Feeling really sad today... miss my friends so much 😢",
    "I can't believe how terrible this service is! So frustrated! 😡",
    "Really nervous about my presentation tomorrow... what if it goes wrong? 😰",
    "OMG! I can't believe I just won the lottery! This is incredible! 😱",
    "Just finished my morning jog. Now heading to the office for meetings."
]

print(f"\n💡 Try these sample posts:")
for i, post in enumerate(sample_posts, 1):
    print(f"{i}. \"{post}\"")

In [ ]:
#14: Model Comparison with Other Algorithms
print("📈 MODEL COMPARISON WITH OTHER ALGORITHMS")
print("=" * 60)

# Import additional algorithms
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# Define models to compare
models = {
    'KNN (Best)': best_model,
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(random_state=42, probability=True),
    'Naive Bayes': MultinomialNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10)
}

model_results = {}

print("🔬 Training and Evaluating Different Algorithms...")
print("-" * 90)
print(f"{'Algorithm':<20} {'Train Acc':<12} {'Test Acc':<12} {'CV Score':<12} {'F1-Score':<12} {'Time':<10}")
print("-" * 90)

# Use numerical features for fair comparison (some algorithms don't handle high-dimensional sparse well)
X_train_comp = X_train_numerical_scaled
X_test_comp = X_test_numerical_scaled

for name, model in models.items():
   start_time = time.time()
   
   if name != 'KNN (Best)':  # KNN is already trained
       # Handle Naive Bayes (requires non-negative features)
       if name == 'Naive Bayes':
           # Scale features to be non-negative for Naive Bayes
           from sklearn.preprocessing import MinMaxScaler
           scaler_nb = MinMaxScaler()
           X_train_nb = scaler_nb.fit_transform(X_train_comp)
           X_test_nb = scaler_nb.transform(X_test_comp)
           model.fit(X_train_nb, y_train)
           train_accuracy = model.score(X_train_nb, y_train)
           test_accuracy = model.score(X_test_nb, y_test)
           y_pred_model = model.predict(X_test_nb)
           cv_scores = cross_val_score(model, X_train_nb, y_train, cv=5)
       else:
           model.fit(X_train_comp, y_train)
           train_accuracy = model.score(X_train_comp, y_train)
           test_accuracy = model.score(X_test_comp, y_test)
           y_pred_model = model.predict(X_test_comp)
           cv_scores = cross_val_score(model, X_train_comp, y_train, cv=5)
   else:
       # For KNN, use the appropriate test set
       if final_accuracy_combined > final_accuracy_numerical:
           train_accuracy = best_model.score(X_train_combined_scaled, y_train)
           test_accuracy = best_model.score(X_test_combined_scaled, y_test)
           y_pred_model = best_model.predict(X_test_combined_scaled)
           cv_scores = cross_val_score(best_model, X_train_combined_scaled, y_train, cv=3)  # Reduced CV for speed
       else:
           train_accuracy = best_model.score(X_train_comp, y_train)
           test_accuracy = best_model.score(X_test_comp, y_test)
           y_pred_model = best_model.predict(X_test_comp)
           cv_scores = cross_val_score(best_model, X_train_comp, y_train, cv=5)
   
   cv_mean = cv_scores.mean()
   f1_macro = f1_score(y_test, y_pred_model, average='macro')
   training_time = time.time() - start_time
   
   model_results[name] = {
       'train_accuracy': train_accuracy,
       'test_accuracy': test_accuracy,
       'cv_score': cv_mean,
       'f1_score': f1_macro,
       'time': training_time,
       'predictions': y_pred_model
   }
   
   print(f"{name:<20} {train_accuracy:.4f}      {test_accuracy:.4f}      {cv_mean:.4f}      {f1_macro:.4f}      {training_time:.3f}s")

# Visualization of model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

model_names = list(model_results.keys())
test_accuracies = [model_results[name]['test_accuracy'] for name in model_names]
f1_scores = [model_results[name]['f1_score'] for name in model_names]
cv_scores = [model_results[name]['cv_score'] for name in model_names]
times = [model_results[name]['time'] for name in model_names]

# Color palette for models
model_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#D63031']

# Test Accuracy Comparison
bars1 = axes[0,0].bar(model_names, test_accuracies, color=model_colors, alpha=0.8)
axes[0,0].set_title('🎯 Test Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_ylim(0, 1)
axes[0,0].tick_params(axis='x', rotation=45)

for bar in bars1:
   height = bar.get_height()
   axes[0,0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# F1-Score Comparison
bars2 = axes[0,1].bar(model_names, f1_scores, color=model_colors, alpha=0.8)
axes[0,1].set_title('⚖️ F1-Score Comparison', fontsize=14, fontweight='bold')
axes[0,1].set_ylabel('F1-Score (Macro)')
axes[0,1].set_ylim(0, 1)
axes[0,1].tick_params(axis='x', rotation=45)

for bar in bars2:
   height = bar.get_height()
   axes[0,1].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# Cross-Validation Score Comparison
bars3 = axes[1,0].bar(model_names, cv_scores, color=model_colors, alpha=0.8)
axes[1,0].set_title('📊 Cross-Validation Score', fontsize=14, fontweight='bold')
axes[1,0].set_ylabel('CV Score')
axes[1,0].set_ylim(0, 1)
axes[1,0].tick_params(axis='x', rotation=45)

for bar in bars3:
   height = bar.get_height()
   axes[1,0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# Training Time Comparison
bars4 = axes[1,1].bar(model_names, times, color=model_colors, alpha=0.8)
axes[1,1].set_title('⏱️ Training Time Comparison', fontsize=14, fontweight='bold')
axes[1,1].set_ylabel('Time (seconds)')
axes[1,1].tick_params(axis='x', rotation=45)

for bar in bars4:
   height = bar.get_height()
   axes[1,1].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                 f'{height:.2f}s', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# Detailed comparison table
print(f"\n📊 DETAILED PERFORMANCE COMPARISON")
print("=" * 80)
comparison_df = pd.DataFrame(model_results).T
comparison_df = comparison_df.drop('predictions', axis=1)  # Remove predictions column for display
comparison_df = comparison_df.round(4)
comparison_df = comparison_df.sort_values('test_accuracy', ascending=False)
print(comparison_df)

# Find best performing models
best_accuracy = comparison_df['test_accuracy'].idxmax()
best_f1 = comparison_df['f1_score'].idxmax()
fastest = comparison_df['time'].idxmin()
most_stable = comparison_df['cv_score'].idxmax()

print(f"\n🏆 ALGORITHM RANKINGS:")
print("=" * 30)
print(f"🎯 Best Test Accuracy: {best_accuracy} ({comparison_df.loc[best_accuracy, 'test_accuracy']:.4f})")
print(f"⚖️ Best F1-Score: {best_f1} ({comparison_df.loc[best_f1, 'f1_score']:.4f})")
print(f"⚡ Fastest Training: {fastest} ({comparison_df.loc[fastest, 'time']:.3f}s)")
print(f"📊 Most Stable (CV): {most_stable} ({comparison_df.loc[most_stable, 'cv_score']:.4f})")

# Confusion matrices comparison for top 3 models
top_3_models = comparison_df.head(3).index.tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, model_name in enumerate(top_3_models):
   y_pred = model_results[model_name]['predictions']
   cm = confusion_matrix(y_test, y_pred)
   
   sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
               xticklabels=class_names, yticklabels=class_names,
               ax=axes[i], cbar=False)
   axes[i].set_title(f'🔍 {model_name}\nAccuracy: {model_results[model_name]["test_accuracy"]:.3f}', 
                    fontweight='bold')
   axes[i].set_xlabel('Predicted')
   axes[i].set_ylabel('True')

plt.tight_layout()
plt.show()

# Algorithm-specific insights
print(f"\n💡 ALGORITHM-SPECIFIC INSIGHTS:")
print("=" * 40)

insights = {
   'KNN (Best)': "• Excellent for capturing local patterns in emotion data\n• Non-parametric, no assumptions about data distribution\n• Performance depends heavily on feature quality",
   'Random Forest': "• Handles mixed feature types well\n• Provides feature importance naturally\n• Robust to outliers and noise",
   'Logistic Regression': "• Fast and interpretable\n• Works well with text features\n• Good baseline for classification",
   'SVM': "• Effective in high-dimensional spaces\n• Memory efficient\n• Can struggle with large datasets",
   'Naive Bayes': "• Fast and simple\n• Works well with text classification\n• Assumes feature independence",
   'Decision Tree': "• Highly interpretable\n• Can capture non-linear patterns\n• Prone to overfitting"
}

for model_name in model_names:
   print(f"\n{model_name}:")
   print(insights.get(model_name, "• General purpose algorithm"))
   print(f"  Test Accuracy: {model_results[model_name]['test_accuracy']:.4f}")
   print(f"  Training Time: {model_results[model_name]['time']:.3f}s")

# Statistical significance test between top models
from scipy import stats

print(f"\n📊 STATISTICAL SIGNIFICANCE ANALYSIS:")
print("=" * 45)

if len(top_3_models) >= 2:
   model1_name = top_3_models[0]
   model2_name = top_3_models[1]
   
   # Get predictions for top 2 models
   pred1 = model_results[model1_name]['predictions']
   pred2 = model_results[model2_name]['predictions']
   
   # Calculate accuracy for each sample
   acc1 = (pred1 == y_test).astype(int)
   acc2 = (pred2 == y_test).astype(int)
   
   # Paired t-test
   t_stat, p_value = stats.ttest_rel(acc1, acc2)
   
   print(f"Comparing {model1_name} vs {model2_name}:")
   print(f"  {model1_name} accuracy: {acc1.mean():.4f}")
   print(f"  {model2_name} accuracy: {acc2.mean():.4f}")
   print(f"  Difference: {acc1.mean() - acc2.mean():.4f}")
   print(f"  P-value: {p_value:.4f}")
   print(f"  Statistically significant: {'Yes' if p_value < 0.05 else 'No'}")

# Final recommendation
print(f"\n🎯 FINAL RECOMMENDATION:")
print("=" * 30)
if best_accuracy == 'KNN (Best)':
   print("✅ KNN performs excellently for emotion detection!")
   print("💡 Reasons:")
   print("  • Captures local patterns in emotional expression")
   print("  • Works well with mixed feature types")
   print("  • No assumptions about data distribution")
   print("  • Interpretable through nearest neighbor examples")
else:
   print(f"✅ {best_accuracy} performs best for this emotion detection task!")
   print(f"💡 However, KNN ranked #{list(comparison_df.index).index('KNN (Best)') + 1} which is still competitive")

print(f"\n🚀 For production use, consider:")
print(f"  • Accuracy: {best_accuracy}")
print(f"  • Speed: {fastest}")
print(f"  • Stability: {most_stable}")
print(f"  • Balance of all factors for your specific use case")